# 07 Advanced Challenge — IC50-style Curve Fitting in Python

## Goal

Fit a simple 4-parameter logistic curve to synthetic dose-response data.

This is more challenging than the earlier dose-response plot.

## Academic boundary

This is a learning example. It is not a production pharmacology workflow.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go

df = pd.read_csv("../data/dose_response_ic50_sample.csv")
df.head()

In [ ]:
summary = (
    df.groupby(["drug_name", "concentration_uM"])
      .agg(mean_viability=("cell_viability_percent", "mean"),
           sd_viability=("cell_viability_percent", "std"),
           n=("cell_viability_percent", "count"))
      .reset_index()
)
summary["sem_viability"] = summary["sd_viability"] / np.sqrt(summary["n"])
summary

In [ ]:
def four_param_logistic(x, bottom, top, ic50, hill):
    return bottom + (top - bottom) / (1 + (x / ic50) ** hill)

fit_results = []

fig = go.Figure()

for drug, group in summary.groupby("drug_name"):
    # Exclude 0 concentration from curve fitting because log/dose curve cannot use x=0 directly.
    fit_group = group[group["concentration_uM"] > 0].copy()
    x = fit_group["concentration_uM"].values
    y = fit_group["mean_viability"].values

    initial_guess = [min(y), max(y), np.median(x), 1.0]
    bounds = ([0, 50, min(x)/10, 0.1], [120, 120, max(x)*10, 5])

    params, _ = curve_fit(
        four_param_logistic,
        x,
        y,
        p0=initial_guess,
        bounds=bounds,
        maxfev=10000
    )

    bottom, top, ic50, hill = params
    fit_results.append({
        "drug_name": drug,
        "bottom": bottom,
        "top": top,
        "estimated_ic50_uM": ic50,
        "hill_slope": hill
    })

    x_pred = np.logspace(np.log10(min(x)), np.log10(max(x)), 200)
    y_pred = four_param_logistic(x_pred, *params)

    fig.add_trace(go.Scatter(
        x=fit_group["concentration_uM"],
        y=fit_group["mean_viability"],
        mode="markers",
        name=f"{drug} observed",
        error_y=dict(type="data", array=fit_group["sem_viability"], visible=True)
    ))

    fig.add_trace(go.Scatter(
        x=x_pred,
        y=y_pred,
        mode="lines",
        name=f"{drug} fitted curve"
    ))

fit_df = pd.DataFrame(fit_results)
fit_df

In [ ]:
fig.update_layout(
    title="IC50-style Curve Fitting Challenge",
    xaxis_title="Concentration (uM, log scale)",
    yaxis_title="Mean Cell Viability (%)",
    template="plotly_white"
)
fig.update_xaxes(type="log")
fig.show()

## Interpretation Practice

1. Which compound has the lower estimated IC50?
2. What does a lower IC50 suggest in this synthetic example?
3. Why should this estimate be treated cautiously?
4. What would improve the reliability of the fit?